# 01.4 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/04-pandas-select-filter/solutions.ipynb)

In [1]:
import pandas as pd

df = pd.DataFrame({
    "Invoice":     ["536365", "536367", "536370", "536370", "C536379", "536381"],
    "Description": ["WHITE HANGING HEART", "ASSORTED COLOUR BIRD", "RED TOADSTOOL LED",
                    "ALARM CLOCK BAKELIKE", "DISCOUNT", "PINK HEART SHAPE"],
    "Quantity":    [6, 32, 24, 48, -1, 12],
    "Price":       [2.55, 1.69, 1.65, 3.75, 27.50, 1.25],
    "Country":     ["United Kingdom", "France", "Germany", "Germany", "Germany", "United Kingdom"],
})
df

,Invoice,Description,Quantity,Price,Country
0,536365,WHITE HANGING HEART,6,2.55,United Kingdom
1,536367,ASSORTED COLOUR BIRD,32,1.69,France
2,536370,RED TOADSTOOL LED,24,1.65,Germany
3,536370,ALARM CLOCK BAKELIKE,48,3.75,Germany
4,C536379,DISCOUNT,-1,27.50,Germany
5,536381,PINK HEART SHAPE,12,1.25,United Kingdom


## Exercise 1

By hand: UK rows are 0 (qty 6) and 5 (qty 12). Only row 5 has quantity ≥ 10 → `PINK HEART SHAPE, 12`.

In [2]:
df.loc[(df["Country"] == "United Kingdom") & (df["Quantity"] >= 10), ["Description", "Quantity"]]

,Description,Quantity
5,PINK HEART SHAPE,12


## Exercise 2

In [3]:
df["revenue"] = df["Quantity"] * df["Price"]
df["is_cancelled"] = df["Invoice"].str.startswith("C")
print("cancelled lines:", df["is_cancelled"].sum())
print("their revenue  :", df.loc[df["is_cancelled"], "revenue"].sum())

cancelled lines: 1
their revenue  : -27.5


## Exercise 3

In [4]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")
if csv.exists():
    retail = pd.read_csv(csv, parse_dates=["InvoiceDate"])
else:
    retail = pd.read_excel(xlsx, sheet_name="Year 2010-2011")
    retail.to_csv(csv, index=False)
retail["Invoice"] = retail["Invoice"].astype(str)        # some invoices start with "C": keep all as text
retail["StockCode"] = retail["StockCode"].astype(str)
print(retail.shape)

(541910, 8)


In [5]:
single = retail[~retail["Invoice"].str.startswith("C") & (retail["Quantity"] == 1)]
single.nlargest(5, "Price")[["Invoice", "Description", "Price"]]

,Invoice,Description,Price
15017,537632,AMAZON FEE,13541.33
299982,A563185,Adjust bad debt,11062.06
173382,551697,POSTAGE,8142.75
297725,562955,DOTCOM POSTAGE,4505.17
268028,560373,Manual,4287.63


Most of these are not products at all ("Manual", "POSTAGE", "Adjust bad debt"): rows that must be removed before any analysis of what customers buy. Lesson 01.5.

## Exercise 4

In [6]:
retail["revenue"] = retail["Quantity"] * retail["Price"]
report = (retail[retail["Country"].isin(["France", "Germany", "Netherlands"])
                 & retail["revenue"].between(20, 100)]
          .sort_values(["Country", "revenue"], ascending=[True, False]))
print(len(report), "rows")
report[["Country", "Invoice", "Description", "revenue"]].head(10)

5870 rows


,Country,Invoice,Description,revenue
393359,France,570823,CHILDRENS CUTLERY DOLLY GIRL,99.6
393360,France,570823,CHILDRENS CUTLERY SPACEBOY,99.6
423714,France,573153,SET OF 4 KNICK KNACK TINS LONDON,99.6
187031,France,552946,TABLECLOTH RED APPLES DESIGN,99.5
498801,France,578541,BROCANTE SHELF WITH HOOKS,99.5
319793,France,564897,PLASTERS IN TIN WOODLAND ANIMALS,99.0
319794,France,564897,PLASTERS IN TIN CIRCUS PARADE,99.0
319795,France,564897,PLASTERS IN TIN SPACEBOY,99.0
509515,France,579345,GREEN METAL BOX ARMY SUPPLIES,99.0
231163,France,557247,REGENCY TEAPOT ROSES,97.8
